# Multithreading & Concurrency

## Thread and Runnable

A thread is a lightweight process. You can create threads by extending `Thread` or implementing `Runnable`.

```java title="ThreadBasics.java"
// Method 1: Extend Thread
class MyThread extends Thread {
    public void run() {
        for (int i = 0; i < 3; i++) {
            System.out.println(Thread.currentThread().getName() + ": " + i);
        }
    }
}

// Method 2: Implement Runnable
class MyRunnable implements Runnable {
    public void run() {
        for (int i = 0; i < 3; i++) {
            System.out.println(Thread.currentThread().getName() + ": " + i);
        }
    }
}

public class ThreadBasics {
    public static void main(String[] args) {
        // Using Thread
        MyThread t1 = new MyThread();
        t1.start();
        
        // Using Runnable
        Thread t2 = new Thread(new MyRunnable());
        t2.start();
    }
}
```

```
Thread-0: 0
Thread-1: 0
Thread-0: 1
Thread-1: 1
Thread-0: 2
Thread-1: 2
```

## Synchronized Methods and Blocks

Synchronization prevents race conditions by ensuring only one thread accesses a resource at a time.

```java title="SynchronizationExample.java"
class Counter {
    private int count = 0;
    
    // Synchronized method
    public synchronized void increment() {
        count++;
    }
    
    public synchronized int getCount() {
        return count;
    }
}

public class SynchronizationExample {
    public static void main(String[] args) throws InterruptedException {
        Counter counter = new Counter();
        
        // Create multiple threads incrementing the counter
        Thread t1 = new Thread(() -> {
            for (int i = 0; i < 1000; i++) {
                counter.increment();
            }
        });
        
        Thread t2 = new Thread(() -> {
            for (int i = 0; i < 1000; i++) {
                counter.increment();
            }
        });
        
        t1.start();
        t2.start();
        t1.join();
        t2.join();
        
        System.out.println("Final count: " + counter.getCount());
    }
}
```

```
Final count: 2000
```

## Volatile Variables

The `volatile` keyword ensures that changes to a variable are visible to all threads immediately.

```java title="VolatileExample.java"
class VolatileFlag {
    private volatile boolean running = true;
    
    public void stop() {
        running = false;
    }
    
    public void work() {
        while (running) {
            // Do work
        }
        System.out.println("Work stopped");
    }
}

public class VolatileExample {
    public static void main(String[] args) throws InterruptedException {
        VolatileFlag flag = new VolatileFlag();
        
        Thread worker = new Thread(flag::work);
        worker.start();
        
        Thread.sleep(100);
        flag.stop();
        
        worker.join();
    }
}
```

```
Work stopped
```

## ExecutorService

`ExecutorService` manages a pool of threads, making it easier to execute tasks concurrently.

```java title="ExecutorServiceExample.java"
import java.util.concurrent.*;

public class ExecutorServiceExample {
    public static void main(String[] args) throws Exception {
        // Create a thread pool with 3 threads
        ExecutorService executor = Executors.newFixedThreadPool(3);
        
        // Submit tasks
        for (int i = 0; i < 5; i++) {
            final int taskId = i;
            executor.submit(() -> {
                System.out.println("Task " + taskId + " running on " + 
                                   Thread.currentThread().getName());
                try {
                    Thread.sleep(1000);
                } catch (InterruptedException e) {
                    e.printStackTrace();
                }
            });
        }
        
        // Shutdown executor
        executor.shutdown();
        executor.awaitTermination(10, TimeUnit.SECONDS);
        System.out.println("All tasks completed");
    }
}
```

```
Task 0 running on pool-1-thread-1
Task 1 running on pool-1-thread-2
Task 2 running on pool-1-thread-3
Task 3 running on pool-1-thread-1
Task 4 running on pool-1-thread-2
All tasks completed
```

## CompletableFuture

`CompletableFuture` provides a way to write asynchronous, non-blocking code with a fluent API.

```java title="CompletableFutureExample.java"
import java.util.concurrent.*;

public class CompletableFutureExample {
    public static void main(String[] args) throws Exception {
        // Create a CompletableFuture
        CompletableFuture<String> future = CompletableFuture.supplyAsync(() -> {
            try {
                Thread.sleep(1000);
            } catch (InterruptedException e) {
                e.printStackTrace();
            }
            return "Result from async task";
        });
        
        // Chain operations
        future.thenApply(result -> result.toUpperCase())
              .thenAccept(result -> System.out.println("Final result: " + result));
        
        // Wait for completion
        future.join();
    }
}
```

```
Final result: RESULT FROM ASYNC TASK
```

## Thread Safety Patterns

Common patterns for ensuring thread safety in concurrent applications.

```java title="ThreadSafetyPatterns.java"
import java.util.concurrent.*;
import java.util.*;

public class ThreadSafetyPatterns {
    // Pattern 1: Immutable objects
    static class ImmutableData {
        private final String value;
        private final int number;
        
        public ImmutableData(String value, int number) {
            this.value = value;
            this.number = number;
        }
        
        public String getValue() { return value; }
        public int getNumber() { return number; }
    }
    
    // Pattern 2: Thread-safe collections
    static class SafeList {
        private List<String> list = Collections.synchronizedList(new ArrayList<>());
        
        public void add(String item) { list.add(item); }
        public List<String> getAll() { return new ArrayList<>(list); }
    }
    
    public static void main(String[] args) throws Exception {
        // Using immutable objects
        ImmutableData data = new ImmutableData("test", 42);
        System.out.println("Immutable: " + data.getValue());
        
        // Using thread-safe collections
        SafeList safeList = new SafeList();
        ExecutorService executor = Executors.newFixedThreadPool(2);
        
        for (int i = 0; i < 5; i++) {
            final int id = i;
            executor.submit(() -> safeList.add("Item " + id));
        }
        
        executor.shutdown();
        executor.awaitTermination(5, TimeUnit.SECONDS);
        System.out.println("Safe list: " + safeList.getAll());
    }
}
```

```
Immutable: test
Safe list: [Item 0, Item 1, Item 2, Item 3, Item 4]
```

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What method must be called to start a thread?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="0">
      <span>run()</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="1">
      <span>start()</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="2">
      <span>execute()</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="3">
      <span>begin()</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What does the synchronized keyword prevent?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8473629" value="0">
      <span>Thread creation</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8473629" value="1">
      <span>Memory leaks</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8473629" value="2">
      <span>Race conditions</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8473629" value="3">
      <span>Exceptions</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What does the volatile keyword ensure?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="0">
      <span>Changes are visible to all threads immediately</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="1">
      <span>Only one thread can access the variable</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="2">
      <span>The variable cannot be modified</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="3">
      <span>Memory is allocated on the stack</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="3">
  <p class="font-semibold mb-3">❓ What does ExecutorService manage?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q6384920" value="0">
      <span>Memory allocation</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q6384920" value="1">
      <span>File I/O operations</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q6384920" value="2">
      <span>Network connections</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q6384920" value="3">
      <span>A pool of threads</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is the main advantage of CompletableFuture?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7293847" value="0">
      <span>It prevents all exceptions</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7293847" value="1">
      <span>It enables asynchronous, non-blocking code</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7293847" value="2">
      <span>It automatically synchronizes threads</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7293847" value="3">
      <span>It eliminates the need for threads</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

Learn more: https://docs.oracle.com/javase/tutorial/essential/concurrency/